# EDA (Exploratory Data Analysis)

EDA on **Global Michelin-Starred Restaurants Dataset**.


1. Load dataset and libraries  
2. Clean data (missing values, duplicates, data types, outliers)  
3. Univariate analysis (single variable distributions)  
4. Bivariate & multivariate analysis (relationships between variables)  
5. Outlier detection (IQR method)  
6. Feature Engineering  
7. Key Insights


## 1. Load Libraries & Dataset

> 🔹 **Note:** Change the CSV file name in the code below if your file has a different name.


In [ ]:
# 1. Load libraries & Dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: nicer plots
sns.set(style="whitegrid")

# Load dataset (change the file name if needed)
df = pd.read_csv("49bde7b6-d328-4707-8e52-c8707c33dc84.csv")

# Preview
df.head()

## 2. Basic Information and Overview

In [ ]:
# Shape of the dataset
print("Shape of dataset:", df.shape)

# Column info
print("\nData Info:")
df.info()

# Summary statistics for numeric & categorical columns
print("\nSummary Statistics (numeric / categorical):")
df.describe(include="all").T

## 3. Missing Value Analysis

In [ ]:
# Missing value count & percentage
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

missing = missing.sort_values(by="missing_percent", ascending=False)
missing

## 4. Duplicate Rows

In [ ]:
# Check for duplicate rows
dup_count = df.duplicated().sum()
print("Number of duplicate rows:", dup_count)

# If any duplicates, remove them
if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates dropped. New shape:", df.shape)
else:
    print("No duplicate rows found.")

## 5. Data Types & Basic Transformations

In [ ]:
# Check data types
df.dtypes

In [ ]:
# Example: convert 'Established_Year' to integer (if not already)
if df["Established_Year"].dtype != "int64":
    df["Established_Year"] = df["Established_Year"].astype(int)

# Example: ensure 'Michelin_Stars' is integer
if df["Michelin_Stars"].dtype != "int64":
    df["Michelin_Stars"] = df["Michelin_Stars"].astype(int)

df.dtypes

## 6. Univariate Analysis

In [ ]:
# List numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", cat_cols)

In [ ]:
# Histograms for numeric features
for col in numeric_cols:
    plt.figure(figsize=(6,4))
    plt.hist(df[col].dropna(), bins=10)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

In [ ]:
# Countplots for categorical features (top categories if many)
for col in cat_cols:
    plt.figure(figsize=(8,4))
    vc = df[col].value_counts()
    vc.plot(kind="bar")
    plt.title(f"Category counts: {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

## 7. Bivariate / Multivariate Analysis

In [ ]:
# Correlation matrix for numeric variables
corr = df[numeric_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap (Numeric Features)")
plt.tight_layout()
plt.show()

In [ ]:
# Example 1: Average price vs Rating
plt.figure(figsize=(6,4))
plt.scatter(df["Average_Price_USD"], df["Rating"])
plt.title("Average Price vs Rating")
plt.xlabel("Average Price (USD)")
plt.ylabel("Rating")
plt.tight_layout()
plt.show()

In [ ]:
# Example 2: Boxplot of Average Price by Price Category
plt.figure(figsize=(8,4))
sns.boxplot(data=df, x="Price_Category", y="Average_Price_USD")
plt.title("Average Price by Price Category")
plt.tight_layout()
plt.show()

In [ ]:
# Example 3: Count of restaurants by Continent and Star Category
plt.figure(figsize=(8,4))
sns.countplot(data=df, x="Continent", hue="Star_Category")
plt.title("Restaurant Count by Continent & Star Category")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Outlier Detection using IQR (Numeric Columns)

In [ ]:
# IQR-based outlier detection function
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

outlier_summary = {}

for col in numeric_cols:
    mask, lower, upper = detect_outliers_iqr(df[col])
    count_outliers = mask.sum()
    outlier_summary[col] = {
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(count_outliers)
    }

pd.DataFrame(outlier_summary).T

In [ ]:
# Optionally: cap outliers (winsorization-style) for a specific column
col = "Average_Price_USD"
mask, lower, upper = detect_outliers_iqr(df[col])

print(f"Outliers in {col} before capping:", mask.sum())

df.loc[df[col] < lower, col] = lower
df.loc[df[col] > upper, col] = upper

mask_after, _, _ = detect_outliers_iqr(df[col])
print(f"Outliers in {col} after capping:", mask_after.sum())

## 9. Feature Engineering

In [ ]:
# Example Feature 1: Price bucket from Average_Price_USD
bins = [0, 150, 300, 500, np.inf]
labels = ["Budget", "Mid-Range", "Premium", "Ultra-Premium"]
df["Price_Bucket"] = pd.cut(df["Average_Price_USD"], bins=bins, labels=labels)

# Example Feature 2: Age of restaurant (same as Years_Operating but recomputed from current year)
CURRENT_YEAR = 2025
df["Computed_Years_Operating"] = CURRENT_YEAR - df["Established_Year"]

# Example Feature 3: Is ultra luxury flag
df["Is_Ultra_Luxury"] = df["Price_Category"].eq("Ultra-Luxury").astype(int)

df.head()

## 10. Key Insights (Example)

**Example Insights **
1. Most restaurants fall under the **Luxury / Ultra-Luxury** price categories with high average prices in USD.  
2. The dataset mainly contains **3-star Michelin restaurants**, with high ratings (mostly above 4.5).  
3. There is a **moderate relationship** between `Average_Price_USD` and `Rating` (you can interpret this from the scatter plot & correlations).  
4. European countries seem to dominate the list of top restaurants (check `Country` and `Continent` distributions).  
5. Older restaurants (higher `Years_Operating`) may or may not have higher ratings – you can verify this using scatter plots.


## 11. Export Cleaned / Engineered Dataset

In [ ]:
# Export the cleaned and feature-engineered data
output_file = "Michelin_Restaurants_Cleaned_FE.csv"
df.to_csv(output_file, index=False)
print(f"Cleaned & engineered data saved as: {output_file}")